# SimpleNews S0–S5 


## 0. Setup and shared-module import

This cell:
1. makes sure a local `src/` package exists (using the shared repo files if they are present beside the notebook),
2. adds the repo root to `sys.path`,
3. imports the shared modules.

In [1]:
from pathlib import Path
import sys
import os

ROOT = Path('.').resolve()
SRC = ROOT / "src"

# Reuse the shared repo modules directly.
# If the repo already exposes a src/ package, use it as-is.
# Otherwise create a lightweight src/ package that LINKS to the root modules
# so we still import the same module files rather than embedding/copying code.
if not SRC.exists():
    SRC.mkdir(parents=True, exist_ok=True)
    (SRC / "__init__.py").write_text("# auto-created package wrapper for shared repo modules\n")
    for name in [
        "config.py",
        "dataset_bootstrap.py",
        "evaluation.py",
        "io_utils.py",
        "pipeline.py",
        "preprocess.py",
        "selectors.py",
        "simplify.py",
    ]:
        src_file = ROOT / name
        dst_file = SRC / name
        if src_file.exists() and not dst_file.exists():
            try:
                os.symlink(src_file, dst_file)
                print(f"Linked {name} -> {src_file.name}")
            except Exception:
                # last-resort fallback if symlinks are unavailable
                import shutil
                shutil.copy2(src_file, dst_file)
                print(f"Copied {name} -> src/{name} (symlink unavailable)")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.config import DEFAULT_CONFIG, Paths
from src.dataset_bootstrap import ensure_dataset
from src.io_utils import ensure_dirs, load_csv, standardize_dataframe
from src.pipeline import generate_system_outputs
from src.preprocess import (
    split_sentences,
    tokenize,
    article_features,
    extract_entities,
    extract_numbers,
    extract_dates,
    token_frequency_rank,
    normalize_whitespace,
)
from src.simplify import conservative_lexical_simplify, protected_spans
from src.evaluation import evaluate_output

print("ROOT:", ROOT)
print("SRC exists:", SRC.exists(), "->", SRC)


ROOT: /Users/gm/Desktop/cs5246_simple_news-main
SRC exists: True -> /Users/gm/Desktop/cs5246_simple_news-main/src


## 1. Install / import extra packages for S4 / S5

In [2]:
import sys, subprocess, importlib.util

REQUIRED = {
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
}

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

In [3]:
import gc
import math
import copy
import warnings
from typing import Dict, List, Optional

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging as hf_logging

# Silence repetitive generation/config warnings that otherwise flood notebook output
hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=r".*max_new_tokens.*max_length.*")
warnings.filterwarnings("ignore", message=r".*min_new_tokens.*min_length.*")
warnings.filterwarnings("ignore", message=r".*forced_bos_token_id.*")
warnings.filterwarnings("ignore", message=r".*tie shared.weight to lm_head.weight.*")

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

def runtime_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = runtime_device()
print("Selected device:", DEVICE)
from tqdm.notebook import tqdm


Selected device: mps


## 2. Configuration

In [4]:
paths = Paths(
    root=ROOT,
    data_dir=ROOT / "data",
    output_dir=ROOT / "outputs",
    figure_dir=ROOT / "outputs/figures",
    table_dir=ROOT / "outputs/tables",
    sample_dir=ROOT / "outputs/samples",
)
ensure_dirs(paths.data_dir, paths.output_dir, paths.figure_dir, paths.table_dir, paths.sample_dir)

# Dataset split to run on
RUN_SPLIT = "validation"   # "train", "validation", or "test"
RUN_LIMIT = None           # set to e.g. 100 for debugging, None for full split

# LLM systems
ENABLE_LLM_SYSTEMS = True

# Model choices
S4_MODEL_NAME = "google/flan-t5-base"         # controlled rewrite
S5_MODEL_NAME = "facebook/bart-large-cnn"     # full-article summarization baseline

# Generation lengths
S4_MAX_NEW_TOKENS = 128
S5_MAX_NEW_TOKENS = 96
S5_MIN_NEW_TOKENS = 48
S5_CHUNK_WORD_BUDGET = 450

# S4 guard
S4_MIN_FACT_RECALL = 0.75

# Save names
RUN_TAG = f"{RUN_SPLIT}_s0_s5_repo_integrated_v16"

print("RUN_SPLIT =", RUN_SPLIT)
print("RUN_LIMIT =", RUN_LIMIT)
print("ENABLE_LLM_SYSTEMS =", ENABLE_LLM_SYSTEMS)
print("S4_MODEL_NAME =", S4_MODEL_NAME)
print("S5_MODEL_NAME =", S5_MODEL_NAME)

RUN_SPLIT = validation
RUN_LIMIT = None
ENABLE_LLM_SYSTEMS = True
S4_MODEL_NAME = google/flan-t5-base
S5_MODEL_NAME = facebook/bart-large-cnn


## 3. Dataset bootstrap + load selected split

In [5]:
dataset_status = ensure_dataset(
    data_dir=paths.data_dir,
    auto_download=True,
    force_download=False,
)
print(dataset_status)

split_path = paths.data_dir / f"{RUN_SPLIT}.csv"
raw_df, colmap = load_csv(split_path)
df = standardize_dataframe(raw_df, colmap)

# Keep doc_id as the repo-native identifier.
# Also expose an id alias for compatibility if someone downstream expects it.
if "doc_id" not in df.columns:
    raise ValueError("Expected repo-standardized dataframe to contain 'doc_id'.")
df["doc_id"] = df["doc_id"].astype(str)
df["id"] = df["doc_id"]

if RUN_LIMIT is not None:
    df = df.iloc[:RUN_LIMIT].copy()

print("Loaded split:", split_path)
print("Rows:", len(df))
df.head(3)


Downloaded CNN/DailyMail and saved local CSV files.
Loaded split: /Users/gm/Desktop/cs5246_simple_news-main/data/validation.csv
Rows: 13368


,doc_id,title,article,reference,id
0,0,,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,0
1,1,,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,1
2,2,,"(CNN)French striker Bafetimbi Gomis, who has a...",Bafetimbi Gomis collapses within 10 minutes of...,2


## 4. Shared I/O helpers

In [6]:
def normalize_pipeline_input(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    if "doc_id" not in out.columns and "id" in out.columns:
        out["doc_id"] = out["id"]
    required = ["doc_id", "article", "reference"]
    for col in required:
        if col not in out.columns:
            raise ValueError(f"Missing required input column: {col}")
    out["doc_id"] = out["doc_id"].astype(str)
    out["id"] = out["doc_id"]  # compatibility alias
    out["article"] = out["article"].fillna("").astype(str)
    out["reference"] = out["reference"].fillna("").astype(str)
    if "title" not in out.columns:
        out["title"] = ""
    return out

def build_pipeline_output_row(
    *,
    row_id: str,
    system: str,
    article: str,
    reference: str,
    output: str,
    extra: Optional[Dict[str, object]] = None,
) -> Dict[str, object]:
    base = {
        "doc_id": row_id,
        "id": row_id,  # compatibility alias
        "system": system,
        "article": article,
        "reference": reference,
        "output": output,
    }
    if extra:
        base.update(extra)
    return base

def append_shared_metrics(out_df: pd.DataFrame) -> pd.DataFrame:
    metrics_rows = []
    for row in out_df.itertuples(index=False):
        metrics = evaluate_output(source=row.article, reference=row.reference, pred=row.output)
        metrics_rows.append(metrics)
    metrics_df = pd.DataFrame(metrics_rows)
    return pd.concat([out_df.reset_index(drop=True), metrics_df.reset_index(drop=True)], axis=1)


## 5. Shared S0–S3 using the repo pipeline

In [7]:
SYSTEM_MAP = {
    "s0_lead3": "S0",
    "s1_textrank": "S1",
    "s2_coverage": "S2",
    "s3_simplified": "S3",
}

def run_shared_s0_s3(df_in: pd.DataFrame) -> pd.DataFrame:
    # Use the shared repo pipeline AS-IS and keep its computed metrics.
    work = df_in.copy()
    shared_df = generate_system_outputs(work[["doc_id", "title", "article", "reference"]], DEFAULT_CONFIG).copy()
    shared_df["doc_id"] = shared_df["doc_id"].astype(str)
    shared_df["system"] = shared_df["system"].map(SYSTEM_MAP)

    # Add the teammate-facing `id` alias without removing repo-native `doc_id`.
    insert_at = list(shared_df.columns).index("doc_id") + 1
    shared_df.insert(insert_at, "id", shared_df["doc_id"])
    return shared_df

df_in = normalize_pipeline_input(df)
shared_outputs = run_shared_s0_s3(df_in)

# Reuse the shared S2 outputs directly for S4 instead of recomputing coverage-aware TextRank.
S2_LOOKUP = (
    shared_outputs.loc[shared_outputs["system"] == "S2", ["doc_id", "output"]]
    .drop_duplicates(subset=["doc_id"])
    .set_index("doc_id")["output"]
    .to_dict()
)

print("Shared outputs rows:", len(shared_outputs))
print("Unique docs in S2 lookup:", len(S2_LOOKUP))
shared_outputs.head(8)

Shared outputs rows: 53472
Unique docs in S2 lookup: 13368


,doc_id,id,system,system_label,title,article,reference,output,replacements,glossary,...,rougel,word_count,sentence_count,avg_sentence_len,flesch_reading_ease,fkgl,entity_coverage,number_coverage,date_coverage,novel_token_ratio
0,0,0,S0,S0 Lead-3,,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,"(CNN)Share, and your gift will be multiplied. ...",[],[],...,0.328358,44,3,14.666667,57.357424,8.902727,0.111111,0.0,0.5,0.0
1,0,0,S1,S1 TextRank,,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,That donor's kidney went to the next recipient...,[],[],...,0.145833,75,3,25.000000,61.892000,10.837333,0.111111,0.0,0.0,0.0
2,0,0,S2,S2 Coverage-aware TextRank,,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,But the power that multiplied Broussard's gift...,[],[],...,0.123894,93,4,23.250000,57.700766,10.987177,0.166667,0.0,0.0,0.0
3,0,0,S3,S3 Coverage-aware + simplification,,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,But the power that multiplied Broussard's gift...,[],[],...,0.123894,93,4,23.250000,57.700766,10.987177,0.166667,0.0,0.0,0.0
4,1,1,S0,S0 Lead-3,,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"(CNN)On the 6th of April 1996, San Jose Clash ...",[],[],...,0.080000,95,4,23.750000,57.164539,11.186184,0.127660,0.1,0.2,0.0
5,1,1,S1,S1 TextRank,,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"According to Phil Rawlins, co-primary owner an...",[],[],...,0.112000,96,4,24.000000,52.050000,11.961667,0.053191,0.0,0.1,0.0
6,1,1,S2,S2 Coverage-aware TextRank,,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"According to Phil Rawlins, co-primary owner an...",[],[],...,0.086957,133,5,26.600000,56.434496,11.996030,0.117021,0.0,0.1,0.0
7,1,1,S3,S3 Coverage-aware + simplification,,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"According to Phil Rawlins, co-primary owner an...",[],[],...,0.086957,133,5,26.600000,56.434496,11.996030,0.117021,0.0,0.1,0.0


## 6. S4 / S5 model helpers

In [8]:
_model_cache = {}

def clear_model_cache():
    global _model_cache
    _model_cache = {}
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

def _clean_generation_config(model, tokenizer):
    gen_cfg = copy.deepcopy(model.generation_config)
    # Remove conflicting defaults that spam warnings when max_new_tokens/min_new_tokens are passed.
    if hasattr(gen_cfg, "max_length"):
        gen_cfg.max_length = None
    if hasattr(gen_cfg, "min_length"):
        gen_cfg.min_length = None
    # Some seq2seq checkpoints expect a BOS id in generation config.
    if getattr(gen_cfg, "forced_bos_token_id", None) is None and getattr(tokenizer, "bos_token_id", None) is not None:
        gen_cfg.forced_bos_token_id = tokenizer.bos_token_id
    return gen_cfg

def load_model_bundle(model_name: str):
    if model_name in _model_cache:
        return _model_cache[model_name]
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    if DEVICE != "cpu":
        model = model.to(DEVICE)
    model.eval()
    model.generation_config = _clean_generation_config(model, tokenizer)
    _model_cache[model_name] = (tokenizer, model)
    return _model_cache[model_name]

def generate_with_model(model_name: str, prompt: str, max_new_tokens: int, min_new_tokens: int = 0) -> str:
    tokenizer, model = load_model_bundle(model_name)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            generation_config=model.generation_config,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            num_beams=4,
            do_sample=False,
            early_stopping=True,
            pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id),
        )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return normalize_whitespace(text)

def fact_recall_against_source(source_text: str, candidate_text: str) -> float:
    source_items = set()
    source_items.update([x.lower() for x in extract_entities(source_text)])
    source_items.update([x.lower() for x in extract_numbers(source_text)])
    source_items.update([x.lower() for x in extract_dates(source_text)])
    if not source_items:
        return 1.0
    cand_items = set()
    cand_items.update([x.lower() for x in extract_entities(candidate_text)])
    cand_items.update([x.lower() for x in extract_numbers(candidate_text)])
    cand_items.update([x.lower() for x in extract_dates(candidate_text)])
    return len(source_items & cand_items) / len(source_items)


## 7. S4 = S2 + constrained LLM rewrite

In [9]:
def build_s4_prompt(selected_text: str) -> str:
    return (
        "Rewrite the following news summary into simpler English.\n"
        "Rules:\n"
        "- Keep names, dates, numbers, money amounts, and percentages.\n"
        "- Do not add new facts.\n"
        "- Use short, clear sentences.\n"
        "- Keep the meaning faithful.\n\n"
        f"Summary:\n{selected_text}\n\n"
        "Simplified summary:"
    )

def get_s2_output_for_row(row: pd.Series) -> str:
    row_id = str(row["doc_id"])
    if row_id in S2_LOOKUP:
        return S2_LOOKUP[row_id]

    # Fallback only if the lookup is missing; normally this path should not run.
    title = row["title"] if "title" in row else ""
    return __import__("src.selectors", fromlist=["coverage_aware_select"]).coverage_aware_select(
        row["article"],
        title=title,
        max_sentences=DEFAULT_CONFIG.selector.max_sentences,
        top_entity_k=DEFAULT_CONFIG.selector.top_entity_k,
    )

def run_s4_on_example(row: pd.Series) -> Dict[str, object]:
    s2_text = get_s2_output_for_row(row)
    prompt = build_s4_prompt(s2_text)
    raw = generate_with_model(S4_MODEL_NAME, prompt, max_new_tokens=S4_MAX_NEW_TOKENS)
    guard = fact_recall_against_source(s2_text, raw)
    if guard < S4_MIN_FACT_RECALL:
        final = s2_text
        fallback_used = True
    else:
        final = raw
        fallback_used = False
    return {
        "output": final,
        "s4_raw": raw,
        "s4_guard_fact_recall": guard,
        "s4_fallback_used": fallback_used,
    }

## 8. S5 = full-article LLM baseline

In [10]:
def chunk_article_by_words(article: str, word_budget: int = S5_CHUNK_WORD_BUDGET) -> List[str]:
    sents = split_sentences(article)
    chunks, cur, cur_words = [], [], 0
    for sent in sents:
        n = len(tokenize(sent))
        if cur and cur_words + n > word_budget:
            chunks.append(" ".join(cur).strip())
            cur, cur_words = [sent], n
        else:
            cur.append(sent)
            cur_words += n
    if cur:
        chunks.append(" ".join(cur).strip())
    return chunks

def build_s5_prompt(article_text: str) -> str:
    return (
        "Summarize this news article in plain English.\n"
        "Rules:\n"
        "- Keep important names, dates, and numbers.\n"
        "- Do not add facts.\n"
        "- Write a short coherent summary.\n\n"
        f"Article:\n{article_text}\n\n"
        "Summary:"
    )

def run_s5_on_example(row: pd.Series) -> Dict[str, object]:
    article = row["article"]
    if len(tokenize(article)) <= S5_CHUNK_WORD_BUDGET:
        out = generate_with_model(
            S5_MODEL_NAME,
            build_s5_prompt(article),
            max_new_tokens=S5_MAX_NEW_TOKENS,
            min_new_tokens=S5_MIN_NEW_TOKENS,
        )
        return {"output": out, "s5_mode": "single_pass"}

    chunks = chunk_article_by_words(article)
    chunk_summaries = []
    for chunk in chunks:
        chunk_out = generate_with_model(
            S5_MODEL_NAME,
            build_s5_prompt(chunk),
            max_new_tokens=max(48, S5_MAX_NEW_TOKENS // 2),
            min_new_tokens=20,
        )
        chunk_summaries.append(chunk_out)

    merged_source = " ".join(chunk_summaries)
    final = generate_with_model(
        S5_MODEL_NAME,
        build_s5_prompt(merged_source),
        max_new_tokens=S5_MAX_NEW_TOKENS,
        min_new_tokens=S5_MIN_NEW_TOKENS,
    )
    return {"output": final, "s5_mode": "chunk_then_merge", "s5_num_chunks": len(chunks)}

## 9. Run S4 / S5 in the shared row format

In [11]:
def run_llm_systems(df_in: pd.DataFrame, show_progress: bool = True, heartbeat_every: int = 25) -> pd.DataFrame:
    rows = []
    total = len(df_in)
    pbar = tqdm(total=total, desc="Running S4/S5", leave=True, mininterval=1.0) if show_progress else None

    try:
        for idx, row in enumerate(df_in.itertuples(index=False), start=1):
            row_series = pd.Series(row._asdict())
            row_id = str(row_series["doc_id"])

            s4_payload = run_s4_on_example(row_series)
            rows.append(build_pipeline_output_row(
                row_id=row_id,
                system="S4",
                article=row_series["article"],
                reference=row_series["reference"],
                output=s4_payload["output"],
                extra={k: v for k, v in s4_payload.items() if k != "output"},
            ))

            s5_payload = run_s5_on_example(row_series)
            rows.append(build_pipeline_output_row(
                row_id=row_id,
                system="S5",
                article=row_series["article"],
                reference=row_series["reference"],
                output=s5_payload["output"],
                extra={k: v for k, v in s5_payload.items() if k != "output"},
            ))

            if pbar is not None:
                pbar.update(1)
                if idx % 5 == 0 or idx == total:
                    pbar.set_postfix_str(f"last_doc={row_id}")

            if heartbeat_every and idx % heartbeat_every == 0:
                print(f"[heartbeat] processed {idx}/{total} articles for S4/S5; last doc_id={row_id}")

    finally:
        if pbar is not None:
            pbar.close()
        clear_model_cache()

    out = pd.DataFrame(rows)
    return append_shared_metrics(out)

if ENABLE_LLM_SYSTEMS:
    print(f"Running LLM systems on {len(df_in)} examples...")
    llm_outputs = run_llm_systems(df_in, show_progress=True, heartbeat_every=25)
else:
    llm_outputs = pd.DataFrame(columns=["doc_id", "id", "system", "article", "reference", "output"])

llm_outputs.head(4) if len(llm_outputs) else "LLM systems disabled"


Running LLM systems on 13368 examples...


Running S4/S5:   0%|          | 0/13368 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

[heartbeat] processed 25/13368 articles for S4/S5; last doc_id=24
[heartbeat] processed 50/13368 articles for S4/S5; last doc_id=49
[heartbeat] processed 75/13368 articles for S4/S5; last doc_id=74
[heartbeat] processed 100/13368 articles for S4/S5; last doc_id=99
[heartbeat] processed 125/13368 articles for S4/S5; last doc_id=124
[heartbeat] processed 150/13368 articles for S4/S5; last doc_id=149
[heartbeat] processed 175/13368 articles for S4/S5; last doc_id=174
[heartbeat] processed 200/13368 articles for S4/S5; last doc_id=199
[heartbeat] processed 225/13368 articles for S4/S5; last doc_id=224
[heartbeat] processed 250/13368 articles for S4/S5; last doc_id=249
[heartbeat] processed 275/13368 articles for S4/S5; last doc_id=274
[heartbeat] processed 300/13368 articles for S4/S5; last doc_id=299
[heartbeat] processed 325/13368 articles for S4/S5; last doc_id=324
[heartbeat] processed 350/13368 articles for S4/S5; last doc_id=349
[heartbeat] processed 375/13368 articles for S4/S5; las

,doc_id,id,system,article,reference,output,s4_raw,s4_guard_fact_recall,s4_fallback_used,s5_mode,...,rougel,word_count,sentence_count,avg_sentence_len,flesch_reading_ease,fkgl,entity_coverage,number_coverage,date_coverage,novel_token_ratio
0,0,0,S4,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,But the power that multiplied Broussard's gift...,But the power that multiplied Broussard's gift...,0.333333,True,NaN,...,0.123894,93,4,23.25,57.700766,10.987177,0.166667,0.00,0.0,0.000000
1,0,0,S5,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,Zully Broussard gave one of her kidneys to a s...,NaN,NaN,NaN,chunk_then_merge,...,0.197183,48,4,12.00,65.992500,7.035833,0.111111,0.00,0.0,0.020833
2,1,1,S4,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"According to Phil Rawlins, co-primary owner an...","""The industry and the game itself has moved on...",0.000000,True,NaN,...,0.086957,133,5,26.60,56.434496,11.996030,0.117021,0.00,0.1,0.000000
3,1,1,S5,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,The first Major League Soccer match was played...,NaN,NaN,NaN,chunk_then_merge,...,0.206897,50,4,12.50,85.859500,4.389000,0.031915,0.25,0.3,0.034483


## 10. Combine all systems and save shared-format outputs

In [12]:
combined = pd.concat([shared_outputs, llm_outputs], ignore_index=True, sort=False)
# keep core columns first
core_cols = ["doc_id", "id", "system", "article", "reference", "output"]
other_cols = [c for c in combined.columns if c not in core_cols]
combined = combined[core_cols + other_cols]

combined_path = paths.output_dir / f"{RUN_TAG}_all_system_outputs.csv"
combined.to_csv(combined_path, index=False)

# Save one CSV per system too
per_system_dir = paths.output_dir / "per_system"
ensure_dirs(per_system_dir)
for sys_name, sub in combined.groupby("system"):
    sub.to_csv(per_system_dir / f"{RUN_TAG}_{sys_name}.csv", index=False)

print("Saved combined outputs to:", combined_path.resolve())
print("Saved per-system CSVs to:", per_system_dir.resolve())
combined.head(12)


Saved combined outputs to: /Users/gm/Desktop/cs5246_simple_news-main/outputs/validation_s0_s5_repo_integrated_v16_all_system_outputs.csv
Saved per-system CSVs to: /Users/gm/Desktop/cs5246_simple_news-main/outputs/per_system


,doc_id,id,system,article,reference,output,system_label,title,replacements,glossary,...,fkgl,entity_coverage,number_coverage,date_coverage,novel_token_ratio,s4_raw,s4_guard_fact_recall,s4_fallback_used,s5_mode,s5_num_chunks
0,0,0,S0,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,"(CNN)Share, and your gift will be multiplied. ...",S0 Lead-3,,[],[],...,8.902727,0.111111,0.000000,0.5,0.0,NaN,NaN,NaN,NaN,NaN
1,0,0,S1,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,That donor's kidney went to the next recipient...,S1 TextRank,,[],[],...,10.837333,0.111111,0.000000,0.0,0.0,NaN,NaN,NaN,NaN,NaN
2,0,0,S2,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,But the power that multiplied Broussard's gift...,S2 Coverage-aware TextRank,,[],[],...,10.987177,0.166667,0.000000,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3,0,0,S3,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,But the power that multiplied Broussard's gift...,S3 Coverage-aware + simplification,,[],[],...,10.987177,0.166667,0.000000,0.0,0.0,NaN,NaN,NaN,NaN,NaN
4,1,1,S0,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"(CNN)On the 6th of April 1996, San Jose Clash ...",S0 Lead-3,,[],[],...,11.186184,0.127660,0.100000,0.2,0.0,NaN,NaN,NaN,NaN,NaN
5,1,1,S1,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"According to Phil Rawlins, co-primary owner an...",S1 TextRank,,[],[],...,11.961667,0.053191,0.000000,0.1,0.0,NaN,NaN,NaN,NaN,NaN
6,1,1,S2,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"According to Phil Rawlins, co-primary owner an...",S2 Coverage-aware TextRank,,[],[],...,11.996030,0.117021,0.000000,0.1,0.0,NaN,NaN,NaN,NaN,NaN
7,1,1,S3,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,"According to Phil Rawlins, co-primary owner an...",S3 Coverage-aware + simplification,,[],[],...,11.996030,0.117021,0.000000,0.1,0.0,NaN,NaN,NaN,NaN,NaN
8,2,2,S0,"(CNN)French striker Bafetimbi Gomis, who has a...",Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",S0 Lead-3,,[],[],...,12.527297,0.257143,0.500000,0.0,0.0,NaN,NaN,NaN,NaN,NaN
9,2,2,S1,"(CNN)French striker Bafetimbi Gomis, who has a...",Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",S1 TextRank,,[],[],...,8.566471,0.228571,0.333333,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## 11. Aggregate comparison table

In [13]:
metric_cols = [
    "rouge1", "rouge2", "rougel",
    "flesch_reading_ease", "fkgl",
    "entity_coverage", "number_coverage", "date_coverage",
    "novel_token_ratio", "word_count", "sentence_count", "avg_sentence_len",
]
summary = combined.groupby("system")[metric_cols].mean().round(4).reset_index()
summary_path = paths.table_dir / f"{RUN_TAG}_metric_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved summary table to:", summary_path.resolve())
summary

Saved summary table to: /Users/gm/Desktop/cs5246_simple_news-main/outputs/tables/validation_s0_s5_repo_integrated_v16_metric_summary.csv


,system,rouge1,rouge2,rougel,flesch_reading_ease,fkgl,entity_coverage,number_coverage,date_coverage,novel_token_ratio,word_count,sentence_count,avg_sentence_len
0,S0,0.3897,0.1726,0.2472,50.5978,12.3383,0.2259,0.2805,0.3530,0.0000,78.2344,3.2348,24.7007
1,S1,0.3390,0.1357,0.2219,51.8746,11.9682,0.2291,0.2247,0.3385,0.0000,88.7922,3.8173,23.9279
2,S2,0.3309,0.1384,0.2149,52.4034,11.8436,0.2971,0.3360,0.4222,0.0000,118.7411,5.1397,23.7231
3,S3,0.3308,0.1382,0.2149,52.5457,11.8237,0.2971,0.3360,0.4222,0.0004,118.7411,5.1397,23.7231
4,S4,0.3343,0.1400,0.2175,52.0518,11.9546,0.2925,0.3330,0.4198,0.0001,114.9098,4.9402,23.9727
5,S5,0.3814,0.1731,0.2718,58.2148,9.4071,0.1722,0.2660,0.3255,0.0220,48.6041,3.0968,17.1783


## 12. One-example preview across all systems

In [14]:
example_id = str(df_in.iloc[0]["doc_id"])
example_rows = combined[combined["doc_id"] == example_id][["system", "output"]].copy()
example_rows


,system,output
0,S0,"(CNN)Share, and your gift will be multiplied. ..."
1,S1,That donor's kidney went to the next recipient...
2,S2,But the power that multiplied Broussard's gift...
3,S3,But the power that multiplied Broussard's gift...
53472,S4,But the power that multiplied Broussard's gift...
53473,S5,Zully Broussard gave one of her kidneys to a s...
